# VLM Baseline

In [2]:
import os
import sys
from pathlib import Path
if Path.cwd().name == "baselines":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

In [3]:
%pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import json
import random
import nltk
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

from helpers.download_data import load_artcap, prepare_artcap
from helpers.metrics import compute_metrics

SEED = 10623
random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.enabled = False

OUTPUT_DIR = Path("data/baseline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PREDICTION_PATH = OUTPUT_DIR / "qwen3_vl_baseline_predictions.csv"

## Download the dataset

In [5]:
prepare_artcap()
test_df = pd.DataFrame(load_artcap(split="test"))
print("Test images:", len(test_df))

Already downloaded and processed at:  data/artcap/captions.jsonl
Test images: 1082


## Load the model

In [6]:
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=224 * 224,
    max_pixels=768 * 768
)
model.eval()

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

Qwen3VLForConditionalGeneration(
  (model): Qwen3VLModel(
    (visual): Qwen3VLVisionModel(
      (patch_embed): Qwen3VLVisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3VLVisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
          (attn): Qwen3VLVisionAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3VLVisionMLP(
            (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
         

## Run the model

In [1]:
PROMPT = """
Describe the visual content of this painting in one concise,
informative sentence.

Focus on:
- The people, objects, and subjects depicted.
- Their actions and spatial relationships.
- Important colors, visual attributes, and setting.

Only describe details that are visually supported by the image.
Do not speculate about the artist, historical context, or
symbolic meaning.

Return only the caption.
"""

In [7]:
def generate_caption(image_path):
    with Image.open(image_path) as source:
        image = source.convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": PROMPT}
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=128, do_sample=False)

    generated_ids = generated_ids[:, inputs.input_ids.shape[1]:]
    caption = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    return caption

In [ ]:
print(generate_caption(test_df.iloc[0]["image_path"]))

In [10]:
predictions = {}
if PREDICTION_PATH.exists():
    saved = pd.read_csv(PREDICTION_PATH, dtype={"image_id": str}, keep_default_na=False)
    predictions = dict(zip(saved["image_id"], saved["pred_caption"]))

def save_predictions():
    results = test_df[["image_id", "image_path"]].copy()
    results["pred_caption"] = results["image_id"].map(predictions)
    results = results.dropna(subset=["pred_caption"])
    temp = PREDICTION_PATH.with_suffix(".tmp")
    results.to_csv(temp, index=False)
    temp.replace(PREDICTION_PATH)

try:
    pending = test_df[~test_df["image_id"].isin(predictions)]
    for i, row in enumerate(tqdm(pending.itertuples(index=False), total=len(pending)), 1):
        predictions[row.image_id] = generate_caption(row.image_path)
        if i % 25 == 0:
            save_predictions()
finally:
    save_predictions()

  0%|          | 0/1082 [00:00<?, ?it/s]

/home/ec2-user/.local/lib/python3.12/site-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
/home/ec2-user/.local/lib/python3.12/site-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


KeyboardInterrupt: 

## Compute Metrics

In [ ]:
captions = test_df["image_id"].map(predictions)
if captions.isna().any():
    raise ValueError("Finish generating test captions before scoring.")

nltk.download("wordnet", quiet=True)

metrics = compute_metrics(
    captions.tolist(),
    test_df["references"].tolist(),
    device=str(model.device),
)
(OUTPUT_DIR / "qwen3_vl_baseline_metrics.json").write_text(json.dumps(metrics, indent=2))
display(pd.Series(metrics, name="Score").to_frame())